# Week 3: Pattern scaling in JAX (Task 6)

A re-implementation of `Week3_PatternScaling.ipynb` in pure JAX, checked against it: the same coefficients, the same
predictions, the same scores. We will build on this in week 8.

Both versions from the numpy notebook:

- **A. Oracle $T$:** patterns $y_\text{cell} = a + b\,T_\text{global}$, driven by the smoothed true `ssp245` global temperature.
- **B. EBM $T$:** the same patterns, driven by a one-box energy-balance model of $T_\text{global}$ from the forcings.

The JAX ideas the brief asks for:

| | used for |
|---|---|
| `jax.jit` | compiling the fits and predictions |
| `jax.grad` | fitting the patterns by gradient descent instead of the closed form |
| `jax.vmap` | running that per-cell fit over all 13,824 grid cells at once |
| `jax.lax.scan` | the EBM's year-by-year lag (a loop that `jit` can compile) |

Run with the **Python (jcm)** kernel. Needs the numpy notebook to have been run first (it reads its saved predictions).

In [1]:
import os
import sys
import time

import jax
jax.config.update("jax_enable_x64", True)    # float64, like numpy; without it JAX defaults to float32
import jax.numpy as jnp
import numpy as np
import pandas as pd
import xarray as xr

REPO = os.path.abspath("..")          # this notebook lives in <repo>/notebooks
assert os.path.exists(f"{REPO}/metrics.py"), "run from the notebooks/ folder"
sys.path.insert(0, REPO)
from metrics import TARGETS, PERIOD, nrmse, global_mean

DATA = os.path.expanduser("~/data/climatebench")
RESULTS = f"{REPO}/results"

TRAIN = ["historical", "hist-GHG", "hist-aer", "ssp126", "ssp370", "ssp585"]   # same split as the numpy version
TEST = "ssp245"
SSPS = [e for e in TRAIN + [TEST] if e.startswith("ssp")]
FORCERS = ["CO2", "CH4", "SO2", "BC"]
TAU = 1          # EBM timescale (years), chosen by leave-one-scenario-out CV in Week3_PatternScaling.ipynb
N_CELLS = 96 * 144
SHORT = {"tas": "tas", "diurnal_temperature_range": "dtr", "pr": "pr", "pr90": "pr90"}

assert TEST not in TRAIN
print("jax", jax.__version__, jax.devices())

jax 0.11.2 [CpuDevice(id=0)]


## 1. Data

Loading and preprocessing stay in xarray/numpy (the same code as the numpy notebook); only the model is in JAX. Everything
that goes into JAX is a plain array:

- `T_train` (753,): global-mean warming of each training year (ensemble mean)
- `Y_train[var]` (753, 13824): each target, one column per grid cell
- `T_oracle` (86,): version A's test input, the 10-year trailing mean of the true `ssp245` global temperature
- `X_forcers[exp]` (years, 4): the EBM inputs, scaled by their training std, with `historical` run first for the SSPs

In [2]:
def load_outputs(exp):
    ds = xr.open_dataset(f"{DATA}/outputs_{exp}.nc")[TARGETS]
    ds["pr"] = ds["pr"] * 86400
    ds["pr90"] = ds["pr90"] * 86400
    return ds


def forcer_series(exp):
    ds = xr.open_dataset(f"{DATA}/inputs_{exp}.nc")
    w = np.cos(np.deg2rad(ds.latitude))
    cols = {"CO2": ds["CO2"].values, "CH4": ds["CH4"].values}
    for name in ["SO2", "BC"]:
        cols[name] = ds[name].weighted(w).mean(["latitude", "longitude"]).values
    return pd.DataFrame(cols, index=ds.time.values)


outputs = {exp: load_outputs(exp) for exp in TRAIN + [TEST]}
truth = outputs[TEST].mean("member")


def gmt(exp):
    return global_mean(outputs[exp]["tas"].mean("member")).values


T_train = jnp.asarray(np.concatenate([gmt(exp) for exp in TRAIN]))
Y_train = {var: jnp.asarray(np.concatenate([outputs[exp][var].mean("member").values.reshape(-1, N_CELLS)
                                            for exp in TRAIN]).astype(np.float64))
           for var in TARGETS}
T_oracle = jnp.asarray(pd.Series(gmt(TEST)).rolling(10, min_periods=1).mean().values)

raw = {exp: forcer_series(exp) for exp in TRAIN + [TEST]}
scale = pd.concat([raw[exp] for exp in TRAIN]).std()
n_years = {exp: len(raw[exp]) for exp in raw}          # the experiment's own years, at the end of its series
X_forcers = {exp: jnp.asarray((pd.concat([raw["historical"], raw[exp]]) if exp in SSPS else raw[exp]).values
                              / scale.values)
             for exp in raw}

print("T_train", T_train.shape, "| Y_train", Y_train["tas"].shape, "| T_oracle", T_oracle.shape)

T_train (753,) | Y_train (753, 13824) | T_oracle (86,)


## 2. Patterns, closed form

Ordinary least squares on $[1, T]$, the same maths as `np.linalg.lstsq` in the numpy notebook, as one jitted function. All
grid cells share the design matrix, so `Y` can have any number of columns.

In [3]:
@jax.jit
def fit_pattern(T, Y):
    A = jnp.stack([jnp.ones_like(T), T], axis=1)
    coef, *_ = jnp.linalg.lstsq(A, Y)
    return coef                          # (2, n_cells): intercept, slope


@jax.jit
def predict_pattern(coef, T):
    return coef[0] + jnp.outer(T, coef[1])            # (n_years, n_cells)


start = time.perf_counter()
fit_pattern(T_train, Y_train["tas"]).block_until_ready()
first = time.perf_counter() - start
start = time.perf_counter()
fit_pattern(T_train, Y_train["pr"]).block_until_ready()
second = time.perf_counter() - start
print(f"first call (traces + compiles): {first:.3f}s, second call (compiled, same shapes): {second:.3f}s")

coef_lstsq = {var: fit_pattern(T_train, Y_train[var]) for var in TARGETS}

first call (traces + compiles): 0.052s, second call (compiled, same shapes): 0.007s


## 3. Patterns, by gradient descent (`grad` + `vmap`)

The same fit, learned instead of solved. `fit_cell` fits **one** grid cell: it minimises the mean squared error of
$a + b\,z$ with `jax.grad`, for a fixed number of steps inside `jax.lax.fori_loop`. `jax.vmap` then maps it over the 13,824
columns of `Y` without a Python loop.

$T$ is standardised to $z = (T - \bar T)/s_T$ first. With $z$ centred and of unit variance, the loss's curvature is the same in
both directions (its Hessian is $2I$), so plain gradient descent converges quickly. Without it, the steps along $a$ and $b$ would
need very different step sizes. The fitted $(a_z, b_z)$ are converted back to the slope and intercept on $T$ at the end.

In [4]:
STEPS, LR = 200, 0.1


def mse(params, z, y):
    a, b = params
    return jnp.mean((a + b * z - y) ** 2)


def fit_cell(z, y):
    step = lambda _, p: p - LR * jax.grad(mse)(p, z, y)
    return jax.lax.fori_loop(0, STEPS, step, jnp.zeros(2))


@jax.jit
def fit_pattern_gd(T, Y):
    mu, sd = T.mean(), T.std()
    z = (T - mu) / sd
    params = jax.vmap(fit_cell, in_axes=(None, 1))(z, Y)    # z shared, one column of Y per cell -> (n_cells, 2)
    a_z, b_z = params[:, 0], params[:, 1]
    slope = b_z / sd
    return jnp.stack([a_z - slope * mu, slope])               # back to (intercept, slope) on T


start = time.perf_counter()
coef_gd = {var: fit_pattern_gd(T_train, Y_train[var]) for var in TARGETS}
jax.block_until_ready(coef_gd)
print(f"gradient-descent fit of all 4 targets x {N_CELLS} cells: {time.perf_counter() - start:.1f}s")

for var in TARGETS:
    print(f"{SHORT[var]:5s} max |gd - lstsq| = {float(jnp.abs(coef_gd[var] - coef_lstsq[var]).max()):.1e}")

gradient-descent fit of all 4 targets x 13824 cells: 4.6s
tas   max |gd - lstsq| = 1.8e-15
dtr   max |gd - lstsq| = 4.4e-16
pr    max |gd - lstsq| = 6.7e-16
pr90  max |gd - lstsq| = 1.4e-15


## 4. The EBM for version B (`lax.scan`)

$T(t) = c_0 + \sum_k c_k\,\tilde x_k(t)$ with $\tilde x(t) = \alpha\,\tilde x(t-1) + (1-\alpha)\,x(t)$, $\alpha = e^{-1/\tau}$.
The lag is a loop over years where each step needs the previous one, so it is written with `jax.lax.scan`, which `jit` can compile
(a Python `for` loop would be unrolled into the compiled program). $\tau$ is the value chosen by cross-validation in the numpy notebook.

In [5]:
@jax.jit
def lag(x, tau):
    a = jnp.exp(-1.0 / tau)
    step = lambda prev, xt: (a * prev + (1 - a) * xt,) * 2      # (new carry, output)
    _, out = jax.lax.scan(step, x[0], x[1:])
    return jnp.concatenate([x[:1], out])


def ebm_features(exp):
    return lag(X_forcers[exp], TAU)[-n_years[exp]:]            # keep the experiment's own years


@jax.jit
def fit_ebm(X, y):
    coef, *_ = jnp.linalg.lstsq(jnp.column_stack([jnp.ones(len(X)), X]), y)
    return coef


X_ebm = jnp.concatenate([ebm_features(exp) for exp in TRAIN])
ebm_coef = fit_ebm(X_ebm, T_train)
T_ebm = ebm_coef[0] + ebm_features(TEST) @ ebm_coef[1:]
print(pd.Series(np.asarray(ebm_coef), index=["c0"] + FORCERS).round(3).to_string())

c0     0.055
CO2    0.781
CH4    0.288
SO2   -0.199
BC    -0.011


## 5. Same scores as the numpy implementation?

Predictions from both JAX fits, for both versions, scored with the same `metrics.py`. They are compared against the files
the numpy notebook saved. Those were stored as float32, so they agree with float64 JAX to about $10^{-6}$ relative, not exactly.

In [6]:
def to_maps(pred, var):
    return xr.DataArray(np.asarray(pred).reshape(-1, 96, 144), dims=truth[var].dims, coords=truth[var].coords)


numpy_files = {"A": xr.open_dataset(f"{RESULTS}/pattern_scaling_ssp245.nc"),
               "B": xr.open_dataset(f"{RESULTS}/pattern_scaling_ebm_ssp245.nc")}
T_test = {"A": T_oracle, "B": T_ebm}

rows, max_diff = {}, {}
for version in ["A", "B"]:
    rows[f"{version}: numpy (saved)"] = {(SHORT[v], k): s for v in TARGETS
                                         for k, s in nrmse(truth[v], numpy_files[version][v]).items()}
    for method, coef in [("lstsq", coef_lstsq), ("grad descent", coef_gd)]:
        preds = {v: to_maps(predict_pattern(coef[v], T_test[version]), v) for v in TARGETS}
        rows[f"{version}: JAX {method}"] = {(SHORT[v], k): s for v in TARGETS
                                            for k, s in nrmse(truth[v], preds[v]).items()}
        max_diff[f"{version}: JAX {method}"] = {
            SHORT[v]: float(np.abs(preds[v] - numpy_files[version][v]).max() / np.abs(numpy_files[version][v]).max())
            for v in TARGETS}

scores = pd.DataFrame(rows).T
scores.round(3)

tas                   dtr                     pr  \
                    Spatial Global  Total Spatial Global   Total Spatial   
A: numpy (saved)      0.080  0.048  0.320   7.977  2.115  18.552   1.902   
A: JAX lstsq          0.080  0.048  0.320   7.977  2.115  18.552   1.902   
A: JAX grad descent   0.080  0.048  0.320   7.977  2.115  18.552   1.902   
B: numpy (saved)      0.088  0.059  0.383   7.983  2.086  18.414   1.936   
B: JAX lstsq          0.088  0.059  0.383   7.983  2.086  18.414   1.936   
B: JAX grad descent   0.088  0.059  0.383   7.983  2.086  18.414   1.936   

                                     pr90                
                    Global  Total Spatial Global  Total  
A: numpy (saved)     0.287  3.338   2.279  0.370  4.128  
A: JAX lstsq         0.287  3.338   2.279  0.370  4.128  
A: JAX grad descent  0.287  3.338   2.279  0.370  4.128  
B: numpy (saved)     0.303  3.452   2.312  0.383  4.226  
B: JAX lstsq         0.303  3.452   2.312  0.383  4.226  
B: JAX grad descent  0.303  3.452   2.312  0.383  4.226

In [7]:
print("largest |JAX - numpy| prediction difference, relative to the largest value of the field:")
print(pd.DataFrame(max_diff).T.map(lambda x: f"{x:.1e}").to_string())

score_diff = (scores.drop(index=[r for r in scores.index if "numpy" in r])
              - np.repeat(scores.loc[["A: numpy (saved)", "B: numpy (saved)"]].values, 2, axis=0))
print(f"\nlargest score difference, any target and metric: {float(score_diff.abs().max().max()):.1e}")
assert float(score_diff.abs().max().max()) < 1e-4

largest |JAX - numpy| prediction difference, relative to the largest value of the field:
                         tas      dtr       pr     pr90
A: JAX lstsq         3.3e-08  4.5e-08  5.1e-08  3.6e-08
A: JAX grad descent  3.3e-08  4.5e-08  5.1e-08  3.6e-08
B: JAX lstsq         3.3e-08  4.5e-08  5.2e-08  3.7e-08
B: JAX grad descent  3.3e-08  4.5e-08  5.2e-08  3.7e-08

largest score difference, any target and metric: 2.9e-08


## 6. Notes

- **Reproduced.** Both JAX fits give the numpy notebook's scores (largest difference printed above), for version A and B.
- **Closed form vs gradient descent.** Identical up to the optimiser's tolerance here, because the problem is a convex least
  squares and standardising $T$ makes it well conditioned. The gradient version is what generalises: in week 8 the same
  `grad`/`vmap` structure can fit models without a closed form.
- **float64.** JAX defaults to float32. Without `jax_enable_x64`, the lstsq slopes differ from numpy's by about
  $2\times10^{-7}$ relative (checked separately). That is harmless for the scores, but the $10^{-15}$ agreement between the two fits
  above is only possible in float64, and float32 errors grow in longer gradient-based training.
- **Ideas for week 8:** the EBM is differentiable end to end (`lag` is a `scan`), so $\tau$ and the forcing coefficients could
  be learned jointly with `jax.grad`, instead of a grid search over $\tau$.